In [1]:
# !pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community

In [2]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("Hugging_face_api_token")
hf_embeddings_api = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",  # lightweight
    # model="Qwen/Qwen3-Embedding-8B",
    # task="feature-extraction",# heavyweight but uses a lot of quota token
    huggingfacehub_api_token=api_key,
)

/Users/kristalshrestha/Documents/Code/LLM_Scratch/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_community.vectorstores import Chroma

In [4]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [5]:
docs=[doc1,doc2,doc3,doc4,doc5]

In [6]:
vector_store=Chroma(
    embedding_function=hf_embeddings_api,
    persist_directory="./Kristal_chrome_db",#location to store chroma database
    collection_name="sample"
)

/var/folders/48/m5h2lpdd1nlc3bhw471fn9c00000gn/T/ipykernel_5647/2530339038.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store=Chroma(


# add documents

In [7]:
vector_store.add_documents(docs)

['0bae2d41-b112-4464-abe3-f630f87f8a77',
 '066e3b25-cd79-43e9-b65c-012f81ed0088',
 '89ccfb6e-82f9-4230-b79e-0921ea603998',
 '750d6468-1e4f-4fbe-936e-1ec89c65edf5',
 '2873aeaa-7fa0-469a-87cf-f682c773e4a7']

# view documents

In [8]:
vector_store.get(include=["embeddings","documents","metadatas"])

{'ids': ['97ac4129-6143-4dc6-b54e-50f5e807ac85',
  'e99433ec-f156-49d5-9092-377b29194023',
  'ddd37346-677b-47cc-a8f9-d5383d99cec2',
  'ce689d55-0cbf-409f-a37e-d4cb37bbf364',
  '099c8d08-1a43-451d-9e19-ef1644b97dad',
  '0bae2d41-b112-4464-abe3-f630f87f8a77',
  '066e3b25-cd79-43e9-b65c-012f81ed0088',
  '89ccfb6e-82f9-4230-b79e-0921ea603998',
  '750d6468-1e4f-4fbe-936e-1ec89c65edf5',
  '2873aeaa-7fa0-469a-87cf-f682c773e4a7'],
 'embeddings': array([[-0.00233748,  0.05902077, -0.04774045, ..., -0.07264046,
          0.00276782, -0.00344092],
        [ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        ...,
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        [ 0.01873968,  0.04382843, 

# search documents


In [9]:
vector_store.similarity_search(
    query="Who among these are a bowler?",
    k=2 # 2 similar documents will be shown
    
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

In [10]:
vector_store.similarity_search(
    query="Who among these are a bowler?",
    k=1 # only 1 which is highest similar document will be shown
    
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.')]

# search with similarity score

In [11]:
vector_store.similarity_search_with_score(
    query="Who among these are a bowler?",
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693602323532104),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693602323532104)]

In [12]:
vector_store.similarity_search_with_score(
    query="Who among these are a bowler?",
    k=1
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693602323532104)]

# meta-data filtering


In [13]:
vector_store.similarity_search_with_score(
    query="",
    filter={"team":"Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-w

# update documents

In [ ]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='97ac4129-6143-4dc6-b54e-50f5e807ac85',#document id of previous which will be replaced 
document=updated_doc1)


# view the changed document

In [15]:
vector_store.get(include=["embeddings","documents","metadatas"])

{'ids': ['97ac4129-6143-4dc6-b54e-50f5e807ac85',
  'e99433ec-f156-49d5-9092-377b29194023',
  'ddd37346-677b-47cc-a8f9-d5383d99cec2',
  'ce689d55-0cbf-409f-a37e-d4cb37bbf364',
  '099c8d08-1a43-451d-9e19-ef1644b97dad',
  '0bae2d41-b112-4464-abe3-f630f87f8a77',
  '066e3b25-cd79-43e9-b65c-012f81ed0088',
  '89ccfb6e-82f9-4230-b79e-0921ea603998',
  '750d6468-1e4f-4fbe-936e-1ec89c65edf5',
  '2873aeaa-7fa0-469a-87cf-f682c773e4a7'],
 'embeddings': array([[-0.00233748,  0.05902077, -0.04774045, ..., -0.07264046,
          0.00276782, -0.00344092],
        [ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        ...,
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        [ 0.01873968,  0.04382843, 

# delete document

In [17]:
vector_store.delete(ids=["97ac4129-6143-4dc6-b54e-50f5e807ac85"])

# view document

In [18]:
vector_store.get(include=["embeddings","documents","metadatas"])

{'ids': ['e99433ec-f156-49d5-9092-377b29194023',
  'ddd37346-677b-47cc-a8f9-d5383d99cec2',
  'ce689d55-0cbf-409f-a37e-d4cb37bbf364',
  '099c8d08-1a43-451d-9e19-ef1644b97dad',
  '0bae2d41-b112-4464-abe3-f630f87f8a77',
  '066e3b25-cd79-43e9-b65c-012f81ed0088',
  '89ccfb6e-82f9-4230-b79e-0921ea603998',
  '750d6468-1e4f-4fbe-936e-1ec89c65edf5',
  '2873aeaa-7fa0-469a-87cf-f682c773e4a7'],
 'embeddings': array([[ 0.0012775 ,  0.03129857, -0.0237538 , ..., -0.0051836 ,
         -0.03280616,  0.02737708],
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        ...,
        [-0.10265914,  0.02650807,  0.02271501, ..., -0.03359747,
         -0.07984945, -0.01507713],
        [ 0.0212339 , -0.02468549, -0.0449436 , ..., -0.10995808,
          0.00572554,  0.09915374],
        [ 0.01873968,  0.04382843, -0.04304254, ..., -0.07801618,
         -0